# v0.24.0 — Geospatial fields

## 1. Intro

SurrealDB stores geometries natively (points, lines, polygons and their collections) and queries them
with its `geo::*` functions and the `INSIDE`, `OUTSIDE` and `INTERSECTS` operators. Since v0.24.0 a
model can declare geometry fields:

* `PointField`, `LineStringField`, `PolygonField`, … (or `GeoField["kind"]`) hold the ORM's
  geometry classes, `GeometryPoint`, `GeometryLine`, `GeometryPolygon`, … Every write stores a native
  SurrealDB geometry and every read gives the class back.
* A geometry field also accepts a GeoJSON object and, for a point, a `(longitude, latitude)` pair;
  `model_dump(mode="json")` writes GeoJSON.
* `nearby()` keeps the records within a distance of a point, `GeoDistance` gives each record its
  distance in metres, and the `inside`, `outside` and `intersects` lookups compare a field with a
  geometry. All of them work in live queries too.

The names are the full SurrealDB-ORM's (`GeoField`, `PointField`, `nearby()`, `GeoDistance`). The
ORM behaves the same on SurrealDB 2.x and 3.x, distances included: this notebook runs top to bottom
on either.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")

# WebSocket: the live query of section 8 needs it.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Models

Three models, one geometry kind each: a restaurant has a point, a zone a polygon, a route a line.
The aliases are nullable with default `None`, as in the full ORM, so `= None` is the usual form.

The notebook starts by dropping its tables, so it can be re-executed in place; on SurrealDB 3.x
removing a missing table raises, hence the suppressed errors.

In [2]:
from surreal_orm_lite import (
    BaseSurrealModel,
    LineStringField,
    PointField,
    PolygonField,
    SurrealConfigDict,
)


class Restaurant(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_restaurant")

    id: str | None = None
    name: str
    cuisine: str
    location: PointField = None


class Zone(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_zone")

    id: str | None = None
    name: str
    area: PolygonField = None


class Route(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_route")

    id: str | None = None
    name: str
    path: LineStringField = None


client = await SurrealDBConnectionManager.get_client()
DEMO_TABLES = ["nb_restaurant", "nb_zone", "nb_route"]


async def reset() -> None:
    """Drop every table of this notebook, whether or not it exists."""
    for table in DEMO_TABLES:
        with contextlib.suppress(Exception):
            await client.query(f"REMOVE TABLE {table};", {})


await reset()
print("Demo tables reset")

Demo tables reset


## 3. Geometry values

A point is `GeometryPoint(longitude, latitude)`, in GeoJSON's order: longitude first. A polygon is
built from its rings (`parse_coordinates()`), or from a GeoJSON object with `Geometry.from_geojson()`.
`to_geojson()` gives the GeoJSON dict back, and `str()` the SurrealQL text of the value. The values
are immutable and hashable, and compare by their coordinates.

In [3]:
from surreal_orm_lite import Geometry, GeometryLine, GeometryPoint, GeometryPolygon

times_square = GeometryPoint(-73.9855, 40.7580)
midtown = GeometryPolygon.parse_coordinates(
    [[(-74.0, 40.745), (-73.97, 40.745), (-73.97, 40.765), (-74.0, 40.765), (-74.0, 40.745)]]
)
downtown = Geometry.from_geojson(
    {
        "type": "Polygon",
        "coordinates": [[[-74.02, 40.70], [-73.97, 40.70], [-73.97, 40.735], [-74.02, 40.735], [-74.02, 40.70]]],
    }
)

print("repr:        ", repr(times_square))
print("to_geojson():", times_square.to_geojson())
print("str():       ", times_square)
print("downtown:    ", type(downtown).__name__, "with", len(downtown.geometry_lines[0].geometry_points), "points")
print("equal to a copy built from GeoJSON:", Geometry.from_geojson(midtown.to_geojson()) == midtown)

repr:         GeometryPoint(longitude=-73.9855, latitude=40.758)
to_geojson(): {'type': 'Point', 'coordinates': [-73.9855, 40.758]}
str():        {type: 'Point', coordinates: [-73.9855, 40.758]}
downtown:     GeometryPolygon with 5 points
equal to a copy built from GeoJSON: True


A geometry field accepts the ORM value, a GeoJSON object and, for a point, a
`(longitude, latitude)` pair; the attribute is always the ORM value. `model_dump()` keeps it, and
`model_dump(mode="json")` writes its GeoJSON object, which `model_validate_json()` reads back.

In [4]:
from_pair = Restaurant(name="Corner Pizza", cuisine="pizza", location=(-73.9877, 40.7547))
from_geojson = Restaurant(
    name="Corner Pizza", cuisine="pizza", location={"type": "Point", "coordinates": [-73.9877, 40.7547]}
)

print("from a pair:            ", repr(from_pair.location))
print("from GeoJSON, equal:    ", from_geojson.location == from_pair.location)
print("model_dump():           ", repr(from_pair.model_dump()["location"]))
print("model_dump(mode='json'):", from_pair.model_dump(mode="json")["location"])
print("JSON round trip, equal: ", Restaurant.model_validate_json(from_pair.model_dump_json()) == from_pair)

from a pair:             GeometryPoint(longitude=-73.9877, latitude=40.7547)
from GeoJSON, equal:     True
model_dump():            GeometryPoint(longitude=-73.9877, latitude=40.7547)
model_dump(mode='json'): {'type': 'Point', 'coordinates': [-73.9877, 40.7547]}
JSON round trip, equal:  True


Invalid values are refused where they are built, before anything is sent: a non-finite
coordinate or a polygon ring of fewer than 4 points raises `InvalidGeometryError` (a `ValueError`),
a `Decimal` coordinate raises `TypeError` (convert it with `float()`), and a geometry field given
one of them raises Pydantic's `ValidationError`.

In [5]:
from decimal import Decimal

from pydantic import ValidationError

from surreal_orm_lite import InvalidGeometryError

try:
    GeometryPolygon.parse_coordinates([[(0.0, 0.0), (1.0, 0.0), (0.0, 0.0)]])
except InvalidGeometryError as exc:
    print("short ring:", exc)

try:
    GeometryPoint(Decimal("-73.9855"), Decimal("40.758"))
except TypeError as exc:
    print("Decimal:   ", exc)

try:
    Restaurant(name="Nowhere", cuisine="none", location=(float("nan"), 0.0))
except ValidationError as exc:
    print("field:     ", exc.errors()[0]["msg"])

short ring: Invalid exterior ring: must have at least 4 points (including closing point), got 3
Decimal:    GeometryPoint longitude must be a real number (int, float or Fraction), not Decimal. Convert a Decimal with float(), or send an SDK geometry: SurrealDB converts its decimal coordinates itself.
field:      Value error, GeometryPoint longitude must be a finite number, got nan


## 4. Save and read back

Five Manhattan restaurants, one in Paris and a food truck without a location, two zones and two
routes. The points are given in each accepted form. SurrealDB stores native geometries (its
`.is_point()` method says so; the food truck has no point), and every read returns the ORM's
classes.

In [6]:
for restaurant in [
    Restaurant(id="pizza", name="Corner Pizza", cuisine="pizza", location=GeometryPoint(-73.9877, 40.7547)),
    Restaurant(id="noodles", name="Bryant Noodles", cuisine="noodles", location=(-73.9832, 40.7536)),
    Restaurant(id="burgers", name="Park Burgers", cuisine="burgers", location=(-73.9881, 40.7417)),
    Restaurant(
        id="deli", name="Village Deli", cuisine="deli", location={"type": "Point", "coordinates": [-73.9969, 40.729]}
    ),
    Restaurant(id="soul", name="Harlem Soul Food", cuisine="soul food", location=GeometryPoint(-73.9442, 40.8116)),
    Restaurant(id="bistro", name="Paris Bistro", cuisine="french", location=GeometryPoint(2.3522, 48.8566)),
    Restaurant(id="truck", name="Food Truck", cuisine="street food"),  # no location
]:
    await restaurant.save()

await Zone(id="midtown", name="Midtown", area=midtown).save()
await Zone(id="downtown", name="Downtown", area=downtown).save()
await Route(
    id="broadway", name="Broadway", path=GeometryLine(GeometryPoint(-73.9897, 40.74), GeometryPoint(-73.9845, 40.76))
).save()
await Route(
    id="champs", name="Champs-Elysees", path=GeometryLine(GeometryPoint(2.2950, 48.8738), GeometryPoint(2.3212, 48.8656))
).save()

stored = await client.query("SELECT name, location.is_point() AS native FROM nb_restaurant ORDER BY name;", {})
print("stored as native points:", {row["name"]: row["native"] for row in stored})

deli = await Restaurant.objects().get("deli")
print("read back:", type(deli.location).__name__, repr(deli.location))
print("equal to the value given:", deli.location == GeometryPoint(-73.9969, 40.729))
print("no location:", (await Restaurant.objects().get("truck")).location)
broadway = await Route.objects().get("broadway")
print("route:", type(broadway.path).__name__, broadway.path)

stored as native points: {'Bryant Noodles': True, 'Corner Pizza': True, 'Food Truck': False, 'Harlem Soul Food': True, 'Paris Bistro': True, 'Park Burgers': True, 'Village Deli': True}
read back: GeometryPoint GeometryPoint(longitude=-73.9969, latitude=40.729)
equal to the value given: True
no location: None
route: GeometryLine {type: 'LineString', coordinates: [[-73.9897, 40.74], [-73.9845, 40.76]]}


## 5. `nearby()`

`nearby(field, point, max_distance)` keeps the records whose field is a point within
`max_distance` metres of `point`, as SurrealDB's `geo::distance` measures it. The point and the
distance are bound as query variables, and a `.is_point()` guard leaves out the records without a
point (the food truck). The condition joins `filter()`, `count()` and the other terminal methods.

In [7]:
near = await Restaurant.objects().nearby("location", times_square, max_distance=2_000).exec()
print("within 2 km of Times Square:", sorted(r.name for r in near))

within_5_km = Restaurant.objects().nearby("location", times_square, max_distance=5_000)
print("within 5 km:                ", await within_5_km.count())
print("within 5 km, burgers:       ", [r.name for r in await within_5_km.filter(cuisine="burgers").exec()])

within 2 km of Times Square: ['Bryant Noodles', 'Corner Pizza', 'Park Burgers']
within 5 km:                 4
within 5 km, burgers:        ['Park Burgers']


## 6. Distance: `GeoDistance`

`annotate(dist=GeoDistance(field, point))` gives each instance the distance as an attribute named
after the alias: metres as a `float`, or `None` when the record has no point. `order_by("dist")`
sorts the nearest first; `order_by("-dist")` the farthest first, the records without a point last.
The attribute is never written back by `update()` or `upsert()`.

In [8]:
from surreal_orm_lite import GeoDistance

closest = await (
    Restaurant.objects()
    .nearby("location", times_square, max_distance=5_000)
    .annotate(dist=GeoDistance("location", times_square))
    .order_by("dist")
    .limit(3)
    .exec()
)
print("the 3 closest:", [(r.name, round(r.dist)) for r in closest])

farthest_first = await Restaurant.objects().annotate(dist=GeoDistance("location", times_square)).order_by("-dist").exec()
for r in farthest_first:
    print(f"  {r.name:<17} {'-' if r.dist is None else f'{r.dist / 1000:,.1f} km'}")
print("not in model_dump():", "dist" not in farthest_first[0].model_dump())

the 3 closest: [('Corner Pizza', 411), ('Bryant Noodles', 526), ('Park Burgers', 1826)]
  Paris Bistro      5,832.9 km
  Harlem Soul Food  6.9 km
  Village Deli      3.4 km
  Park Burgers      1.8 km
  Bryant Noodles    0.5 km
  Corner Pizza      0.4 km
  Food Truck        -
not in model_dump(): True


## 7. Spatial lookups

`inside`, `outside` and `intersects` compare a field with a geometry through SurrealQL's `INSIDE`,
`OUTSIDE` and `INTERSECTS`, and the existing `contains` lookup tells whether a polygon field
contains a point. The geometry is bound as one value; the lookups work in `filter()` and in `Q`
objects.

In [9]:
async def names(queryset) -> list[str]:
    """The names of the records a queryset returns, sorted."""
    return sorted(r.name for r in await queryset.exec())


print("restaurants inside Midtown:      ", await names(Restaurant.objects().filter(location__inside=midtown)))
print("restaurants inside Downtown:     ", await names(Restaurant.objects().filter(location__inside=downtown)))
print("zones that contain Times Square: ", await names(Zone.objects().filter(area__contains=times_square)))
print("routes that intersect Midtown:   ", await names(Route.objects().filter(path__intersects=midtown)))

restaurants inside Midtown:       ['Bryant Noodles', 'Corner Pizza']
restaurants inside Downtown:      ['Village Deli']
zones that contain Times Square:  ['Midtown']
routes that intersect Midtown:    ['Broadway']


`outside` keeps SurrealQL's meaning: it is true for every record whose field is not inside the
geometry, including a record without a location. That is why the food truck is outside Midtown;
`~Q(location__inside=midtown)` returns the same records.

In [10]:
from surreal_orm_lite import Q

outside = await names(Restaurant.objects().filter(location__outside=midtown))
not_inside = await names(Restaurant.objects().filter(~Q(location__inside=midtown)))
print("outside Midtown:     ", outside)
print("~Q(inside), the same:", not_inside == outside)

outside Midtown:      ['Food Truck', 'Harlem Soul Food', 'Paris Bistro', 'Park Burgers', 'Village Deli']
~Q(inside), the same: True


## 8. Live query

`nearby()` and the spatial lookups work in `live()` and `watch()`. A live query cannot carry bound
variables on SurrealDB 2.x, so the ORM writes the point into the statement as a GeoJSON object
literal, with the same notifications on both lines. A restaurant in Paris is saved first (outside
the 2 km), then one near Times Square: only the second is notified, as a typed `Restaurant`.
`asyncio.timeout` keeps the notebook from hanging on a quiet stream.

In [11]:
import asyncio

async with Restaurant.objects().nearby("location", times_square, 2_000).live() as stream:
    await Restaurant(id="crepes", name="Seine Creperie", cuisine="french", location=GeometryPoint(2.35, 48.86)).save()
    await Restaurant(id="cart", name="Hot Dog Cart", cuisine="street food", location=GeometryPoint(-73.9851, 40.7589)).save()
    async with asyncio.timeout(5):
        event = await anext(stream)

print(event.action, event.record_id)
print("instance:", type(event.instance).__name__, event.instance.name)
print("location:", repr(event.instance.location))

CREATE nb_restaurant:cart
instance: Restaurant Hot Dog Cart
location: GeometryPoint(longitude=-73.9851, latitude=40.7589)


## 9. Cleanup

Drop the demo tables and close the connection.

In [12]:
await reset()
await SurrealDBConnectionManager.close_connection()
print("Done")

Done
